# Indexer check

Runs the photo-map indexer on your real photos and shows what to eyeball.
Keep this notebook in the `notebooks` folder of the project.

The index and thumbnails go into the project's `data/` folder, shared with the app. Delete that folder any time to start over; your photos are only ever read, never changed.

**Checks:**
1. Full scan runs without errors (note the time)
2. Portrait photos are upright in thumbnails
3. Duplicates are real duplicates (no different photos merged)
4. Suggested locations look plausible on the map
5. Rescan is near-instant
6. Adding and deleting a photo is picked up

In [ ]:
%pip install --quiet pillow pillow-heif tzdata pandas folium

In [ ]:
# Make the photomap package importable from this notebooks folder
import sys
from pathlib import Path

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

## Settings

In [ ]:
from pathlib import Path

PHOTO_FOLDER = Path(r"C:\Users\you\Pictures\photo-map-test")  # keep the r before the quotes
DATA_DIR = None         # None = the project's data folder, shared with the app
WORKERS = 4              # photos read in parallel

In [ ]:
import base64
import sqlite3
import time

import pandas as pd
from IPython.display import HTML, display

from photomap import db
from photomap.config import DEFAULT_DATA_DIR
from photomap.indexer import Settings, index_folder

settings = Settings(PHOTO_FOLDER, DATA_DIR or DEFAULT_DATA_DIR, workers=WORKERS)


def query(sql, params=()):
    with sqlite3.connect(settings.db_path) as conn:
        return pd.read_sql_query(sql, conn, params=params)


def thumb_tag(thumb_file, width=140, caption=""):
    if not isinstance(thumb_file, str):
        return f"<div style='width:{width}px;font-size:11px'>no thumbnail<br>{caption}</div>"
    data = base64.b64encode((settings.thumb_dir / thumb_file).read_bytes()).decode()
    return (f"<div style='display:inline-block;margin:4px;vertical-align:top;width:{width}px;"
            f"font-size:11px;word-break:break-all'><img src='data:image/jpeg;base64,{data}' "
            f"style='max-width:{width}px;max-height:{width}px'><br>{caption}</div>")


def grid(df, caption_col="file_name", limit=40):
    if df.empty:
        display(HTML("<i>nothing to show</i>"))
        return
    display(HTML("".join(thumb_tag(r.thumb_file, caption=str(getattr(r, caption_col)))
                         for r in df.head(limit).itertuples())))

## 1. Full scan
The first run reads every photo; expect a few minutes for ~1,500 photos. Progress prints every 100.

In [ ]:
def progress(done, total):
    if done % 100 == 0 or done == total:
        print(f"read {done:,} of {total:,}", flush=True)

report = index_folder(settings, progress)
print()
print(report)
if report.new:
    print(f"\n{report.seconds / report.new * 1000:.0f} ms per photo "
          f"-> about {report.seconds / report.new * 50_000 / 60:.0f} min for 50,000 photos")
for path, err in report.errors[:20]:
    print("unreadable:", path, "-", err)

## 2. Portrait photos upright?
Every thumbnail below should be the right way up. Sideways ones mean the rotation handling needs work.

In [ ]:
portraits = query("SELECT file_name, thumb_file, orientation FROM photos "
                  "WHERE error IS NULL AND height > width ORDER BY RANDOM() LIMIT 24")
print(f"{len(portraits)} sample portrait photos; orientation tags seen:",
      query("SELECT orientation, COUNT(*) n FROM photos GROUP BY orientation").to_dict("records"))
grid(portraits)

## 3. Duplicates
Each row: the photo that's kept (left) and the copy hidden behind it. They should be the same picture. The survey suggests around 9 pairs, plus some stripped copies.

In [ ]:
dups = query("""
    SELECT d.file_name AS hidden, d.thumb_file AS hidden_thumb, d.lat IS NOT NULL AS hidden_gps,
           k.file_name AS kept,   k.thumb_file AS kept_thumb,   k.lat IS NOT NULL AS kept_gps
    FROM photos d JOIN photos k ON k.id = d.duplicate_of
    ORDER BY k.file_name""")
print(f"{len(dups)} duplicates hidden")
html = []
for r in dups.itertuples():
    html.append("<div style='border-bottom:1px solid #ccc;padding:4px'>"
                + thumb_tag(r.kept_thumb, caption=f"KEPT {r.kept} {'(GPS)' if r.kept_gps else '(no GPS)'}")
                + thumb_tag(r.hidden_thumb, caption=f"hidden {r.hidden} {'(GPS)' if r.hidden_gps else '(no GPS)'}")
                + "</div>")
display(HTML("".join(html) or "<i>no duplicates found</i>"))

## 4. Suggested locations
Photos without GPS that were taken within 5 minutes of a photo with GPS. Left: the photo needing a location. Right: the photo it borrows from. On the map, orange dots are suggestions, joined to the blue photo they came from.

In [ ]:
sugg = query("""
    SELECT p.id, p.file_name, p.thumb_file, p.taken_local, p.suggested_lat, p.suggested_lon,
           ROUND(p.suggested_gap_s / 60.0, 1) AS gap_min,
           s.file_name AS source_file, s.thumb_file AS source_thumb, s.lat AS source_lat, s.lon AS source_lon
    FROM review_queue q JOIN photos p ON p.id = q.id JOIN photos s ON s.id = p.suggested_from
    ORDER BY p.taken_local""")
print(f"{len(sugg)} photos have a suggested location")
display(HTML("".join(
    "<div style='border-bottom:1px solid #ccc;padding:4px'>"
    + thumb_tag(r.thumb_file, caption=f"{r.file_name}<br>{r.taken_local}")
    + thumb_tag(r.source_thumb, caption=f"from {r.source_file}<br>{r.gap_min} min apart")
    + "</div>" for r in sugg.head(15).itertuples())))

In [ ]:
import folium

if not sugg.empty:
    m = folium.Map(location=[sugg.suggested_lat.mean(), sugg.suggested_lon.mean()], zoom_start=11)
    for r in sugg.itertuples():
        folium.PolyLine([[r.source_lat, r.source_lon], [r.suggested_lat, r.suggested_lon]], weight=1).add_to(m)
        folium.CircleMarker([r.source_lat, r.source_lon], radius=4, color="#2a6fdb", fill=True,
                            tooltip=f"GPS: {r.source_file}").add_to(m)
        folium.CircleMarker([r.suggested_lat, r.suggested_lon], radius=6, color="#e8710a", fill=True,
                            tooltip=f"suggested: {r.file_name} ({r.gap_min} min)").add_to(m)
    display(m)

### Still needing a location
Photos with no GPS and no nearby photo to borrow from. In the app these go to manual review.

In [ ]:
left = query("SELECT file_name, thumb_file, taken_local FROM review_queue WHERE suggested_lat IS NULL ORDER BY file_name")
print(f"{len(left)} photos need placing by hand")
grid(left)

## 5. Rescan speed
Nothing changed, so this should take a second or two and read 0 photos.

In [ ]:
again = index_folder(settings)
print(again)

## 6. Adding and deleting a photo
In File Explorer: copy one new photo into the folder and delete (or move out) another. Then run this cell. Expect `1 new` and `1 removed`. Put things back afterwards and run it again to see them reversed.

In [ ]:
print(index_folder(settings))

## Map of everything indexed
A quick look; the real app will be the Dash map.

In [ ]:
from folium.plugins import MarkerCluster

points = query("SELECT file_name, taken_local, lat, lon, location_source FROM map_points")
m = folium.Map(location=[points.lat.mean(), points.lon.mean()], zoom_start=10)
cluster = MarkerCluster().add_to(m)
for r in points.itertuples():
    folium.Marker([r.lat, r.lon], tooltip=f"{r.file_name} {r.taken_local or ''}").add_to(cluster)
display(m)

## Results to send back
Paste this output into the chat.

In [ ]:
summary = query("""
    SELECT COUNT(*) AS photos,
           SUM(error IS NOT NULL) AS unreadable,
           SUM(error IS NULL AND thumb_ok = 0) AS thumb_failures,
           SUM(duplicate_of IS NOT NULL) AS duplicates,
           (SELECT COUNT(*) FROM map_points) AS on_map,
           (SELECT COUNT(*) FROM review_queue) AS needs_review,
           (SELECT COUNT(*) FROM review_queue WHERE suggested_lat IS NOT NULL) AS with_suggestion
    FROM photos""").iloc[0].to_dict()
summary["first_scan_seconds"] = round(report.seconds, 1)
summary["rescan_seconds"] = round(again.seconds, 2)
print(summary)